# acquire

> pull the web, papers, video, files, code and JSON APIs into the vault — once, or on a schedule

In [ ]:
#| default_exp acquire

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Every method here is one [fossick](https://github.com/vedicreader/fossick) call plus `Vault.add`.
fossick knows how to get past bot walls, read arXiv and YouTube, and sniff a page's JSON API; the
vault's job is only to file what comes back with the provenance that explains why it is there.

In [ ]:
#| export
import json, re, time, uuid, warnings
from urllib.parse import urlparse
from fastcore.all import AttrDict, L, Path, first, patch
from fossick import json_records
from litesearch import code_exts, dir2files, pdf_parse, DOC_EXTS
from vishalakshi.core import Vault, KINDS, is_sanskrit_file

In [ ]:
#| export
def clip(s:str, n:int=120) -> str:
    'Collapse whitespace and clip a scraped title to something a breadcrumb can carry.'
    return re.sub(r'\s+', ' ', (s or '').strip())[:n] or 'untitled'

def md_title(md:str, fallback:str='') -> str:
    "First markdown heading in `md`, else `fallback` — scraped <title>s are often junk."
    m = re.search(r'^#{1,2} +(.+)$', md or '', flags=re.M)
    return clip(m.group(1) if m else fallback)

`auto=True` is the default on `url` because a bot wall returns HTTP 200 with a challenge page,
which would otherwise be indexed as if it were the article.

`web` is the loop the vault exists for: the query that found a page is kept in its metadata, so
months later `sources()` still says *why* a document is in your corpus. Results already present are
skipped rather than duplicated, so re-running an overlapping search is cheap.

In [ ]:
#| export
@patch
def url(self:Vault,
        url:str,            # page to read
        title:str=None,     # defaults to the page's first heading, else its path
        sel:str=None,       # CSS selector to narrow the page before conversion
        kind:str='web',
        auto:bool=True,     # escalate plain -> heavy -> stealthy -> logged-in Chrome past bot walls
        meta:dict=None,
        force:bool=False,
        **kw                # forwarded to fossick.fetch (verify=, headers=, heavy=…)
) -> dict:
    'Fetch one URL, convert it to markdown and file it in the vault.'
    from fossick import fetch, to_md
    pg = fetch(url, sel=sel, auto=auto, **kw)
    md, st = (to_md(pg, sel=sel) if pg is not None else ''), getattr(pg, 'status', None)
    if not md.strip() or (st or 200) >= 400:            # a failed fetch is None, and a bot wall is a 4xx
        return dict(url=url, skipped=f'could not read the page (status {st})', status=st)
    m = dict(meta or {}, url=url, status=st, fetched_at=time.time())
    return dict(self.add(md, title or md_title(md, urlparse(url).path.rsplit('/', 1)[-1] or url),
                         source=url, kind=kind, meta=m, force=force), url=url)

@patch
def crawl(self:Vault, start_url:str, max_pages:int=10, sel:str=None, **kw) -> L:
    'Crawl a docs site or blog from a start URL and file every page in the vault.'
    from fossick import crawl as _crawl, to_md
    pgs = L((pg.url, to_md(pg, sel=sel)) for pg in _crawl(start_url, sel=sel, max_pages=max_pages, **kw))
    return L(self.add(md, md_title(md, u), source=u, kind='web',
                      meta=dict(url=u, via='crawl', root=start_url, fetched_at=time.time()))
             for u, md in pgs if md.strip())

@patch
def web(self:Vault,
        query:str,          # what to search for
        n:int=5,            # top results to read
        google:bool=False,  # real Google ranking via a stealth browser (slower)
        chars:int=60000,    # max markdown chars kept per source
        **kw                # forwarded to fossick.research
) -> AttrDict:
    'Search the web, read the top `n` results, and file all of them in the vault.'
    from fossick import research
    res = research(query, n=n, engine='google' if google else 'search', chars=chars, **kw)
    srcs = L(res['sources']).filter(lambda s: s['md'].strip() and s['href'])
    added = srcs.map(lambda s: dict(self.add(s['md'], clip(s['title']), source=s['href'], kind='web',
                                             meta=dict(url=s['href'], query=query, fetched_at=time.time())),
                                    url=s['href']))
    # what `research` could not read is the difference between "the web is quiet on this" and "five
    # pages bot-walled us", and only one of those is worth re-running
    return AttrDict(query=query, n_found=len(res['sources']), added=added,
                    dropped=L(res.get('dropped') or []))

In [ ]:
#| eval: false
# Live-scraped search: it needs the network, it is rate-limited, and it returns nothing when
# throttled — which then fails the *next* cell on an empty index. A demo, not a test, so it is
# not run during `nbdev_prepare`; the dispatch it exercises is tested against `what_is` below.
v=Vault(':memory:')
v.web('what is consciousness?', google=True)

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-08-11 06:57:52] INFO: Fetched (200) <GET https://www.google.com/search?q=what+is+consciousness%3F&hl=en&num=17&sei=TDt6ap_5Fsqc4-EP1fOf2AM> (referer: https://www.google.com/)
[2026-08-11 06:57:53] INFO: Fetched (200) <GET https://en.wikipedia.org/wiki/Consciousness> (referer: https://www.google.com/)
[2026-08-11 06:57:53] INFO: Fetched (200) <GET https://pmc.ncbi.nlm.nih.gov/articles/PMC5924785/> (referer: https://www.google.com/)
[2026-08-11 06:57:53] INFO: Fetched (200) <GET https://mcgovern.mit.edu/2024/04/29/what-is-consciousness/> (referer: https://www.google.com/)
[2026-08-11 06:57:53] INFO: Fetched (200) <GET https://www.reddit.com/r/askphilosophy/comments/kar7as/what_is_consciousness_exactly_a

```python
{ 'added': [{'doc_id': '97aaa07e0c0a3b44', 'title': 'Consciousness', 'kind': 'web', 'nodes': 9, 'chunks': 140, 'url': 'https://en.wikipedia.org/wiki/Consciousness'}, {'doc_id': '0d51d0f4dd5d84f6', 'title': 'What is consciousness exactly and why do so many people ...', 'kind': 'web', 'nodes': 2, 'chunks': 3, 'url': 'https://www.reddit.com/r/askphilosophy/comments/kar7as/what_is_consciousness_exactly_and_why_do_so_many/'}, {'doc_id': '9b5b9514d7b82195', 'title': 'Human Consciousness: Where Is It From and What Is It for', 'kind': 'web', 'nodes': 18, 'chunks': 135, 'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5924785/'}, {'doc_id': '872731547af5c248', 'title': 'What Is Consciousness?', 'kind': 'web', 'nodes': 2, 'chunks': 47, 'url': 'https://www.nature.com/articles/d41586-018-05097-x'}, {'doc_id': '84b9a0d41ce5faa8', 'title': 'What is consciousness? - MIT McGovern Institute', 'kind': 'web', 'nodes': 7, 'chunks': 19, 'url': 'https://mcgovern.mit.edu/2024/04/29/what-is-consciousness/'}],
  'dropped': [],
  'n_found': 5,
  'query': 'what is consciousness?'}
```

In [ ]:
#| eval: false
v.db.t.store.fts_search('consciousness')[0]['content']

"Some philosophers believe that Block's two types of consciousness are not the end of the story. William Lycan, for example, argued in his book _Consciousness and Experience_ that at least eight clearly distinct types of consciousness can be identified (organism consciousness; control consciousness; consciousness _of_ ; state/event consciousness; reportability; introspective consciousness; subjective consciousness; self-consciousness)—and that even this list omits several more obscure forms.[57]\n\n"

### Routing: what a target *is*

`what_is` names the kind and `grab` dispatches on it. A paper goes to the shelf a science encoder
wrote, a repo — local or on GitHub — splits between the vault and kosha through `add_tree`, and
everything unrouted stays put. `shelf=` overrides the route.

`crawl` is a flag rather than a seventh kind because no URL says whether you want the page or the
site: `what_is` would have to guess, and a wrong guess quietly fetches ten pages.

fossick shells out to git and keeps the checkout in its own cache, and `fossick.gh_clone` hands that
working tree over — so nothing here downloads or copies anything. This module used to infer the
checkout from a file `read_gh_repo` happened to return, which raised `no files in <url>` for any repo
matching none of its default globs: a JS or Go repo could not be filed at all. A lone source file is
worth more to kosha than to the chunk store, so
`gh_file` hands the code branch to `index_code` — against the clone itself, not a copy of the one
file. A file indexed out of its package resolves to `core.create` where the same file indexed in
place is `fastlite.core.create`, and the clone is already on disk and already current, so the
context is free. `code()` is the shallow path — files as documents — and `index_code` is the real
one.

In [ ]:
#| export
@patch
def arxiv(self:Vault, id_or_url:str, save_dir:str=None, force:bool=False, **kw) -> dict:
    'Read an arXiv paper (metadata + full text) into the vault as `kind="arxiv"`.'
    if (v := self.route('arxiv')) is not self: return v.arxiv(id_or_url, save_dir=save_dir, force=force, **kw)
    from fossick import read_arxiv
    p = read_arxiv(id_or_url, save_dir=save_dir or str(self.assets('pdfs')), force=force, **kw)
    md = f"# {p['title']}\n\n{p.get('summary','')}\n\n{p.get('source') or ''}"
    m=dict(authors=list(p.get('authors') or []),published=p.get('published'),pdf_path=p.get('pdf_path'),fetched_at=time.time())
    return self.add(md, clip(p['title']), source=p.get('link') or id_or_url, kind='arxiv', force=force,meta=m)

@patch
def pdf(self:Vault, path_or_url:str, title:str=None, force:bool=False, **kw) -> dict:
    'Read a PDF (local path or URL) into the vault, one tree node per heading.'
    from fossick import get_pdf
    if (p:=Path(path_or_url)).exists(): return self.add_file(p, title=title, kind='pdf', force=force)
    if (doc := get_pdf(path_or_url, **kw)) is None: return dict(source=path_or_url, skipped='not a PDF or could not be fetched')
    stem = path_or_url.rsplit('/', 1)[-1].split('?')[0]
    return self.add(list(enumerate(pdf_parse(doc, out_path=self.assets(stem or 'pdf')))),
                    title or clip(stem), source=path_or_url, kind='pdf', force=force,
                    meta=dict(url=path_or_url, fetched_at=time.time()))

@patch
def youtube(self:Vault, url:str, force:bool=False) -> dict:
    "Read a YouTube video's transcript and metadata into the vault."
    from fossick import read_yt
    v = read_yt(url, force=force)
    if not (v.get('source') or '').strip(): return dict(source=url,skipped='no transcript', title=v.get('title'))
    md = f"# {v['title']}\n\n{v.get('description','')}\n\n## Transcript\n\n{v['source']}"
    meta=dict(url=url, channel=v.get('channel'), duration=v.get('duration'), upload_date=v.get('upload_date'), fetched_at=time.time())
    return self.add(md, clip(v['title']), source=url, kind='youtube', force=force,meta=meta)

_GH_BLOB = re.compile(r'https?://github\.com/([^/]+)/([^/]+)/blob/[^/]+/(.+)')

@patch
def github(self:Vault, url:str, **kw) -> AttrDict:
    'File a whole GitHub repo: prose to the vault, source to kosha, through `add_tree`.'
    from fossick import gh_clone
    return self.add_tree(gh_clone(url), **kw)

@patch
def gh_file(self:Vault, url:str, title:str=None, **kw) -> dict:
    'File one file from GitHub: prose into the vault, source into a kosha-indexed directory.'
    from fossick import read_gh_file
    if not (m := _GH_BLOB.match(url)): return dict(source=url, skipped='not a GitHub file URL')
    owner, repo, path = m.groups()
    p = Path(path)
    if p.suffix.lower() in code_exts.split(','):
        from fossick import gh_clone
        d = gh_clone(f'https://github.com/{owner}/{repo}')
        return dict(source=url, kind='code', path=str(d/path), code=self.index_code(d))
    return dict(self.add(read_gh_file(url), title or clip(p.name), source=url, kind='web',
                         meta=dict(url=url, repo=f'{owner}/{repo}', fetched_at=time.time()), **kw), url=url)

def what_is(target:str) -> str:
    """Which kind of thing a `grab` target names: `dir`, `file`, `sanskrit`, `arxiv`, `youtube`, `github`, `ghfile`, `pdf` or `web`.

    `sanskrit` is checked before the generic `file` because it is the one local case whose *shelf*
    depends on what is inside the file rather than on its extension: a GRETIL edition and an
    ordinary web page are both `.htm`. Naming it here is what lets `route` send it to an encoder
    that can read it, and the shelf has to be chosen before the document is read, not after."""
    if (p:=Path(target)).is_dir(): return 'dir'
    if p.exists(): return 'sanskrit' if is_sanskrit_file(p) else 'file'
    if 'arxiv.org' in target or re.fullmatch(r'\d{4}\.\d{4,5}(v\d+)?', target): return 'arxiv'
    if re.search(r'youtube\.com|youtu\.be', target): return 'youtube'
    if re.match(r'https?://github\.com/[^/]+/[^/]+', target): return 'ghfile' if '/blob/' in target else 'github'
    if target.lower().split('?')[0].endswith('.pdf'): return 'pdf'
    if target.startswith('http'): return 'web'
    raise ValueError(f'not a URL, an arXiv id, a file or a directory: {target}')

@patch
def grab(self:Vault,
         target:str,        # a URL, an arXiv id, a YouTube link, a GitHub repo or file, a PDF, a local file or a directory
         title:str=None,
         sel:str=None,      # CSS selector, for the web cases
         shelf:str=None,    # shelf to file it on; None -> whichever `KIND_SHELF` names for its kind
         crawl:bool=False,  # follow links from a web target instead of reading the one page
         max_pages:int=10,  # pages to visit when crawling
         **kw               # forwarded to whichever method the target names
):
    'File anything, by looking at what it is — the one call a CLI or an agent needs.'
    kind = what_is(target)
    v = self.shelf(shelf) if shelf else self.route(kind)
    if crawl:              return v.crawl(target, max_pages=max_pages, sel=sel, **kw)
    if kind == 'dir':      return v.add_tree(target, **kw)
    if kind in ('file', 'sanskrit'): return v.add_file(target, title=title, **kw)
    if kind == 'arxiv':    return v.arxiv(target, **kw)
    if kind == 'youtube':  return v.youtube(target, **kw)
    if kind == 'github':   return v.github(target, **kw)
    if kind == 'ghfile':   return v.gh_file(target, title=title, **kw)
    if kind == 'pdf':      return v.pdf(target, title=title, **kw)
    return v.url(target, title=title, sel=sel, **kw)

@patch
def code(self:Vault, dir:str, types:str=code_exts, **kw) -> L:
    "File a source tree into the vault as `kind='code'`, so code and prose answer one query."
    return self.add_dir(dir, types=types, kind='code', **kw)

In [ ]:
#| hide
# What a target *is* decides which method reads it, so the table is worth testing directly.
test_eq(what_is('https://github.com/AnswerDotAI/fastcore'), 'github')
test_eq(what_is('https://github.com/AnswerDotAI/fastcore/tree/master/nbs'), 'github')
test_eq(what_is('https://github.com/AnswerDotAI/fastcore/blob/master/fastcore/basics.py'), 'ghfile')
test_eq(what_is('https://github.com/AnswerDotAI'), 'web')          # a user page is not a repo
test_eq(what_is('https://arxiv.org/abs/1706.03762'), 'arxiv')
test_eq(what_is('https://example.com/paper.pdf'), 'pdf')

# The checkout is fossick's to hand over: `gh_clone` returns the working tree, and a local path
# passes through. We used to infer that path from a file `read_gh_repo` happened to return, which
# raised `no files in <url>` for any repo matching none of its default globs — a JS or Go repo.
from fossick import gh_clone
from subprocess import run as _run
from tempfile import mkdtemp
rd = Path(mkdtemp()).resolve()
(rd/'pkg').mkdir()
(rd/'pkg'/'m.py').write_text('def f(): return 1\n')
_run(['git', 'init', '-q', str(rd)], check=True)
test_eq(gh_clone(str(rd)), rd)
(rd/'pkg'/'m.py').unlink()                                         # nothing kosha would index
test_eq(gh_clone(str(rd)), rd)                                     # still the checkout

# `grab` dispatches on the kind, and `crawl=True` overrides it for any web target
calls = []
saved = {n: getattr(Vault, n) for n in ('crawl', 'github', 'gh_file', 'url')}
for n in saved: setattr(Vault, n, (lambda n: lambda self, *a, **kw: calls.append((n, a, kw)))(n))
try:
    g = Vault(':memory:', offline=True)
    g.grab('https://github.com/AnswerDotAI/fastcore');                  test_eq(calls[-1][0], 'github')
    g.grab('https://github.com/AnswerDotAI/fastcore/blob/master/x.py'); test_eq(calls[-1][0], 'gh_file')
    g.grab('https://example.com/post');                                 test_eq(calls[-1][0], 'url')
    g.grab('https://example.com/docs/', crawl=True, max_pages=3)
    test_eq(calls[-1][0], 'crawl')
    test_eq(calls[-1][2]['max_pages'], 3)
finally:
    for n, f in saved.items(): setattr(Vault, n, f)

# `gh_file` splits on the extension: prose is a document, source is something for kosha to index
import fossick
_saved = fossick.read_gh_file
fossick.read_gh_file = lambda url: '# Title\n\nfused ranks share no vector space\n'
try:
    vg = Vault(str(Path(mkdtemp()).resolve()/'v.db'), offline=True)
    r = vg.gh_file('https://github.com/o/r/blob/main/docs/README.md')
    test_eq(r['kind'], 'web')
    assert vg.search('fused ranks')
    test_eq(vg.gh_file('https://github.com/o/r')['skipped'], 'not a GitHub file URL')
    # the source branch never fetches: the file is already in the clone fossick keeps current, and
    # indexing it there is what lets kosha resolve `pkg.g` rather than a bare `g`
    rr = Path(mkdtemp()).resolve()/'r'
    (rr/'pkg').mkdir(parents=True)
    (rr/'pkg'/'g.py').write_text('def g():\n    return 2\n')
    _run(['git', 'init', '-q', str(rr)], check=True)
    _gc, fossick.gh_clone = fossick.gh_clone, lambda url: rr
    try:
        r = vg.gh_file('https://github.com/o/r/blob/main/pkg/g.py')
        test_eq(r['kind'], 'code')
        test_eq(Path(r['path']), rr/'pkg'/'g.py')          # the file in the checkout, not a copy
        assert Path(r['path']).read_text().startswith('def g')
        assert r['code']                                    # kosha indexed the repo, not one orphan
    finally: fossick.gh_clone = _gc
finally: fossick.read_gh_file = _saved

parse files from /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpd4fetc3m/r: 100%|██████████| 1/1 [00:00<00:00, 1250.17it/s]


One directory, two indexes. `add_dir` files documents and `index_code` fills kosha; a tree usually
holds both, and remembering to call each is exactly the sort of thing a library should do for you.

Prose wants chunks, headings and embeddings. Code wants an AST: `symbol()`, `where_to_add()` and the
call graph exist only in kosha, and filing a `.py` into the prose store as a document — which is what
`Vault.code()` does — buys none of them. So `add_dir` handles the documents, `index_code` handles the
source, and `add_tree` is the one call that knows which is which.

The graph is rebuilt once at the end rather than per file, for the reason `poll` rebuilds it once per
tick: `connect()` reads the whole store, so a tree of two hundred files would otherwise pay for it
two hundred times. If kosha is not installed the source is filed as prose instead and `code` says so
— a searchable fallback beats a traceback. `code` is a plain switch rather than a three-way choice
because forcing kosha at a tree with no source in it would index nothing at the price of loading a
code embedder.

In [ ]:
#| export
@patch
def add_tree(self:Vault,
             dir:str,                # tree to ingest
             types:str=DOC_EXTS,     # extensions filed into the vault as prose
             code:bool=True,         # index source files with kosha, when the tree has any
             kind:str=None,          # override the kind for the prose half
             connect:bool=False,     # rebuild the entity graph at the end — see below
             verbose:bool=False,
             **kw                    # forwarded to add_file
) -> AttrDict:
    '''Ingest a whole tree, each half to the index that can actually answer questions about it.

    `connect` is off because it is not incremental: it rebuilds the graph over the *whole* vault, so
    leaving it on made the tenth tree cost as much as the first nine. Nothing in the default
    retrieval path reads the graph either — `context(graph=True)` is the caller that does. Ingest
    what you have, then call `connect()` once.'''
    p = Path(dir)
    if not p.is_dir(): raise ValueError(f'not a directory: {dir}')
    docs = self.add_dir(p, types=types, kind=kind, **kw)
    srcs = dir2files(p, types=code_exts) if code else L()
    out = AttrDict(dir=str(p), docs=docs, n_docs=len(docs), n_code=len(srcs), code=None)
    if srcs:
        try: out.code = self.index_code(p, verbose=verbose)
        except Exception as e:
            warnings.warn(f'could not index {len(srcs)} source files with kosha '
                          f'({type(e).__name__}: {str(e)[:120]}) — filing them as prose instead, so '
                          f'they are at least searchable. Install kosha for symbol search.')
            out.code = dict(error=f'{type(e).__name__}: {str(e)[:200]}', filed_as_prose=len(srcs))
            out.docs = docs + srcs.map(self.add_file, kind='code', **kw)
    if connect and (out.n_docs or out.code): out.graph = self.connect()
    return out

### Harvest: read a page's API, not its HTML

Listing, product and dashboard pages render from an internal JSON API. Reading that API is faster,
paginates cleanly and survives redesigns, where scraping the DOM does none of those. One document
per harvest, one `##` section per record — so `build_tree` gives every record its own node and
breadcrumb, and a catalogue becomes individually retrievable rows sitting next to your notes.

APIs bury their payload at different depths (`results`, `data.products.items`, a bare array), so
fossick's `json_records` walks for the longest list of dicts rather than guessing a key name — and
tells you which key held them, which is what pagination then reads. Both halves used to search
differently: `apis` counted rows anywhere in the payload and picked the endpoint on that count, while
`paginate_api` looked only at the top level, so a nested list was 24 records at `pages=1` and "no
records found" at `pages=2`. Re-harvesting the same source replaces rather than duplicates when
`force=True`, and is otherwise a no-op — which is what makes `harvest` safe to put behind a `watch`.

In [ ]:
#| export
def records_md(recs, title_keys=('name', 'title', 'displayName', 'productName', 'label', 'sku', 'id')) -> str:
    'Records as markdown, one `##` section per record, so each becomes its own retrievable node.'
    def ttl(r): return clip(next((r[k] for k in title_keys if isinstance(r.get(k), str) and r[k].strip()),
                                 json.dumps(r, default=str)), 80)
    return '\n\n'.join(f'## {ttl(r)}\n\n```json\n{json.dumps(r, indent=1, default=str)}\n```'
                       for r in L(recs).map(lambda r: r if isinstance(r, dict) else dict(value=r)))

In [ ]:
#| export
@patch
def apis(self:Vault,
         url:str,             # page to watch
         pattern:str='*',     # glob/regex filtering captured request URLs
         session:bool=False,  # capture through the logged-in debug Chrome
         preview:int=240,     # chars of each response shown
         **kw                 # forwarded to fossick.find_xhr
) -> L:
    'Discover the JSON endpoints a page calls, so you can read its data instead of its HTML.'
    from fossick import find_xhr
    self._caps = L(find_xhr(url, pattern=pattern, session=session, **kw))
    return L(AttrDict(n=i, url=h['url'], content_type=h.get('content_type'),
                      records=len(json_records(h.get('data'))),
                      preview=json.dumps(h.get('data'), default=str)[:preview])
             for i, h in enumerate(self._caps))

@patch
def harvest(self:Vault,
            url:str,                # the page whose API you want
            pattern:str='*',        # which captured request URLs to keep
            title:str=None,         # document title; defaults to the page host + path
            capture:int=None,       # replay a specific endpoint from the last apis() call
            pages:int=1,            # pages to pull; >1 paginates the endpoint
            page_field:str='page',  # query/body key incremented per page
            session:bool=False,     # capture through the logged-in Chrome
            force:bool=False,
            **kw
) -> dict:
    "Sniff a page's JSON API, pull the records, and file them in the vault as `kind='data'`."
    from fossick import replay_xhr, paginate_api
    caps = getattr(self, '_caps', None)
    if capture is None or not caps:
        found = self.apis(url, pattern=pattern, session=session)
        if not found: return dict(url=url, skipped='no JSON endpoints captured')
        capture, caps = max(found, key=lambda h: h.records).n, self._caps
    cap, hit = caps[capture].get('capture'), caps[capture]
    ep = (cap or hit)['url']
    try:one, key = json_records(replay_xhr(cap, **kw).json() if cap else hit.get('data'), with_key=True)
    except Exception as e: return dict(url=url, endpoint=ep, skipped=f'{type(e).__name__}: {e}')
    if pages > 1 and one: kw.setdefault('page_size', len(one))
    items = (paginate_api(ep, payload=cap.get('request_body') if cap else None, page_field=page_field,
      results_field=key, method=(cap.get('method') or 'GET').upper() if cap else 'GET', max_pages=pages, **kw)
             if pages > 1 else one)
    if not items: return dict(url=url, endpoint=ep, skipped='no records found in the response')
    ttl = title or f'{urlparse(url).netloc}{urlparse(url).path}'.strip('/')
    return dict(self.add_records(items, ttl, source=ep, force=force, meta=dict(page=url, endpoint=ep,
           harvested_at=time.time())), endpoint=ep, records=len(items))

@patch
def add_records(self:Vault, recs:list, title:str, source:str=None, kind:str='data', force:bool=False,
                meta:dict=None) -> dict:
    'File a list of dicts you already have (any API, any export) as one document, one section each.'
    return self.add(f'# {title}\n\n{records_md(recs)}', title, source=source or f'records:{title}',
                    kind=kind, force=force, meta=dict(meta or {}, records=len(L(recs))))

In [ ]:
#| hide
# One page and many pages have to find the same list. The rows here are nested under `data.products`,
# which is where `apis` counted them — while pagination looked only at the top-level keys and found
# nothing, so `pages=1` filed 3 records and `pages=2` reported "no records found in the response".
import fossick
from fossick import json_records
_pay = {'meta': {'total': 3}, 'data': {'products': [{'sku': 'A'}, {'sku': 'B'}, {'sku': 'C'}]}}
_cap = dict(url='https://shop.example/api/products', method='GET', request_body=None)

_h = Vault(':memory:', offline=True)
_h._caps = [dict(url=_cap['url'], data=_pay, capture=_cap)]
_seen = {}
_sv = fossick.replay_xhr, fossick.paginate_api
fossick.replay_xhr = lambda cap, **kw: AttrDict(json=lambda: _pay)
fossick.paginate_api = lambda url, **kw: (_seen.update(kw), _pay['data']['products'])[1]
try:
    r1 = _h.harvest(_cap['url'], capture=0)
    test_eq(r1['records'], 3)
    r2 = _h.harvest(_cap['url'], capture=0, pages=2, force=True)
    test_eq(r2['records'], 3)                       # the same rows, not a skipped run
    test_eq(_seen['results_field'], 'products')     # pagination is told which list to read
finally: fossick.replay_xhr, fossick.paginate_api = _sv

test_eq(_h.doc(_cap['url'])['meta']['records'], 3)  # and each row is its own section in the vault
assert _h.search('sku')

# A session-less sniff records the *response* but not the request, so `find_xhr` returns no
# `capture` at all — and refusing to file the rows it already holds made every default-path
# harvest report `skipped`, on a page whose records `apis` had just printed.
_n = Vault(':memory:', offline=True)
_n._caps = [dict(url=_cap['url'], data=_pay)]        # exactly what `session=False` leaves behind
_r = _n.harvest('https://shop.example/listing', capture=0, title='sniffed')
test_eq((_r['records'], _r['endpoint']), (3, _cap['url']))
assert _n.search('sku')

# ...and page 2 is asked for at the size the first response actually was. `paginate_api` assumes 24,
# so a 3-row page looks like the last one and pagination stops before it starts.
_seen.clear()
_sv2 = fossick.paginate_api
fossick.paginate_api = lambda url, **kw: (_seen.update(kw), _pay['data']['products'])[1]
try: _n.harvest('https://shop.example/listing', capture=0, title='sniffed', pages=3, force=True)
finally: fossick.paginate_api = _sv2
test_eq((_seen['page_size'], _seen['max_pages'], _seen['method']), (3, 3, 'GET'))

### Watches: keeping it current

A watch is what turns the vault from an archive into something that stays current. `action` names
an acquisition method, so anything you can file once you can file on a schedule; `remind` writes a
note instead of fetching, which is the recurring-reminder case with no network involved. `poll()`
is the tick — cron, a scheduler, or a frontend button.

`run_watch` records a failure on the row and returns it rather than raising: one dead URL must not
stop a polling loop from servicing every other watch. `poll` rebuilds the entity graph once at the
end rather than per watch, because `connect()` reads the whole store and a poll that fired five
watches would otherwise pay for it five times.

In [ ]:
#| export
ACTIONS = ('url', 'web', 'harvest', 'arxiv', 'youtube', 'crawl', 'remind')

_DUR, _MULT = re.compile(r'([\d.]+)\s*([smhdw])', re.I), dict(s=1, m=60, h=3600, d=86400, w=604800)

def secs(every) -> float:
    "Seconds from `'30m'`, `'6h'`, `'2 days'`, `'1w'`, `'1h30m'`, or a number of seconds."
    try: return float(every)
    except (TypeError, ValueError): pass
    if not (ms := _DUR.findall(str(every))): raise ValueError(f'not a duration: {every!r}')
    return sum(float(n) * _MULT[u.lower()] for n, u in ms)

@patch
def _w(self:Vault):
    'The watches table, created on first use.'
    t = self.db.t.watches
    t.create(id=str, action=str, target=str, params=str, every=float, note=str, enabled=int,
             last_run=float, last_status=str, next_run=float, runs=int, pk='id', if_not_exists=True)
    return t

@patch
def watch(self:Vault,
          target:str,         # URL, query, arXiv id, or the text of a reminder
          action:str='url',   # one of ACTIONS — what to do when it fires
          every:str='1d',     # interval: '30m', '6h', '1d', '1w', or seconds
          note:str=None,      # why you are watching
          start:float=None,   # first run time (epoch); defaults to now
          **params            # forwarded to the action (n=, pattern=, pages=, sel=, ...)
) -> dict:
    'Register a recurring job: re-read a page, re-run a search, re-harvest an API, or remind you.'
    assert action in ACTIONS, f'action must be one of {ACTIONS}'
    row = dict(id=uuid.uuid4().hex[:12], action=action, target=target, params=json.dumps(params),
               every=secs(every), note=note or '', enabled=1, next_run=start or time.time(), runs=0)
    self._w().insert(row, replace=True)
    return dict(row, params=params)

@patch
def watches(self:Vault, due_only:bool=False, at:float=None) -> L:
    'Every registered watch, soonest first; `due_only` keeps the ones whose next run has arrived.'
    where = f'enabled=1 AND next_run<={at or time.time()}' if due_only else None
    return L(self._w()(where=where, order_by='next_run')).map(
        lambda r: dict(r, params=json.loads(r['params'] or '{}')))

@patch
def unwatch(self:Vault, watch_id:str):
    'Delete a watch. The documents it already filed stay in the vault.'
    self._w().delete(watch_id)

@patch
def pause(self:Vault, watch_id:str, enabled:bool=False):
    'Disable (or re-enable) a watch without losing it.'
    self._w().update(dict(id=watch_id, enabled=int(enabled)))

@patch
def run_watch(self:Vault, w:dict) -> dict:
    'Fire one watch and record the outcome.'
    t0 = time.time()
    try:
        res = (self.note(w['target'], title=w.get('note') or None, tags=['reminder'])
               if w['action'] == 'remind' else getattr(self, w['action'])(w['target'], **w['params']))
        status = 'skipped' if isinstance(res, dict) and res.get('skipped') else 'ok'
    except Exception as e: res, status = dict(error=f'{type(e).__name__}: {str(e)[:200]}'), 'error'
    now = time.time()
    self._w().update(dict(id=w['id'], last_run=now, last_status=status, runs=w['runs']+1,
                          next_run=now + w['every']))
    return dict(watch_id=w['id'], action=w['action'], target=w['target'], status=status,
                took=round(now-t0, 2), result=res)

@patch
def poll(self:Vault, at:float=None, limit:int=None, connect:bool=True) -> dict:
    'Run every watch that is due. This is the tick a scheduler, a cron or a frontend calls.'
    ran = self.watches(due_only=True, at=at)[:limit].map(self.run_watch)
    if connect and ran.filter(lambda r: r['status'] == 'ok'): self.connect()
    pending = self.watches()
    return dict(checked=len(pending), ran=len(ran), results=ran,
                next_due=pending[0]['next_run'] if pending else None)

## Try it

In [ ]:
v = Vault(':memory:')
v.add_records([dict(sku='A1', name='Free range eggs', price=4.5),
               dict(sku='B2', name='Oat milk', price=2.1)], 'dairy')
v.search('eggs')[0]['breadcrumb']

'dairy › Free range eggs'

In [ ]:
# the records walk is fossick's — the same one `paginate_api` uses, so `harvest(pages=1)` and
# `harvest(pages=5)` cannot disagree about which list in a payload holds the rows
_pay = {'data': {'items': [{'a': 1}, {'a': 2}, {'a': 3}]}}
test_eq(len(json_records(_pay)), 3)
test_eq(json_records(_pay, with_key=True)[1], 'items')
test_eq(secs('6h'), 21600); test_eq(secs('1w'), 604800); test_eq(secs(90), 90)
# every documented spelling, and no pandas: it was never a declared dependency, so a clean install
# raised ImportError the first time anyone set a watch on a schedule rather than a bare number
test_eq(secs('30m'), 1800); test_eq(secs('2 days'), 172800); test_eq(secs('45s'), 45)
test_eq(secs('1h30m'), 5400); test_eq(secs('90'), 90)
test_fail(lambda: secs('whenever'), contains='not a duration')
w = v.watch('late chunking', action='web', every='1d', n=3)
test_eq(w['params'], dict(n=3))
test_eq(len(v.watches(due_only=True)), 1)
v.unwatch(w['id'])
test_eq(len(v.watches()), 0)

In [ ]:
#| hide
# a mixed tree — a README and a source file, the shape of any repo
from tempfile import mkdtemp
d = Path(mkdtemp())
(d/'README.md').write_text('# fuse\n\nRanks are fused because the legs share no vector space.')
(d/'fuse.py').write_text('def fuse(ranks):\n    "Reciprocal rank fusion over ranked lists."\n    return ranks\n')

r = Vault(':memory:').add_tree(d, connect=False)
test_eq((r.n_docs, r.n_code), (1, 1))                 # the doc to the vault, the source to kosha
test_eq(r.docs.attrgot('title'), ['README'])
assert r.code and not r.code.get('error'), r.code     # kosha answered, not the prose fallback

# code=False leaves the source where it is, and a tree with no source never reaches kosha at all
r = Vault(':memory:').add_tree(d, code=False, connect=False)
test_eq((r.n_code, r.code), (0, None))
test_fail(lambda: Vault(':memory:').add_tree(d/'README.md'), contains='not a directory')

# the graph is rebuilt once at the end, not once per file
r = Vault(':memory:').add_tree(d, connect=True)
assert r.graph['entities'] > 0, r.graph

parse files from /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpeekzjq36:   0%|          | 0/1 [00:00<?, ?it/s]

parse files from /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpeekzjq36: 100%|██████████| 1/1 [00:00<00:00, 671.63it/s]


In [ ]:
#| hide
# `arxiv` routes itself, because the *method* fixes the kind and `grab` would route it anyway
v2 = Vault(':memory:')
test_eq(v2.route('arxiv').name, 'papers')
# ...and nothing else does. `find` is a single-shelf primitive, so a write that quietly lands on
# another shelf is a read that quietly returns nothing — `code`'s whole purpose is that code and
# prose answer one query, and `add_records` takes its kind as an argument rather than in its name.
v2.add_records([dict(sku='A1', name='free range eggs')], 'groceries')
test_eq(v2.doc('records:groceries')['title'], 'groceries')       # right here, where `find` looks
assert v2.search('free range eggs')
(d/'b.py').write_text('def fuse(x):\n    return x\n')
v2.code(d)
assert 'b' in v2.sources(kind='code').attrgot('title')   # filed here, as `kind='code'` prose
# an explicit shelf is never overruled — what was asked for is what happens
test_eq(v2.shelf('sanskrit', offline=True).route('arxiv').name, 'sanskrit')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()